In [ ]:

# The constants were blindly generated from my notes so I can run the simulation, they need to be checked and probably changed (density seems a bit too high for plastic :) 
# --- Material ---
RHO = 1240.0          # density rho (PLA)               [kg/m^3]
C_P = 1800.0          # specific heat c (PLA)           [J/(kg K)]
KAPPA = 0.13          # thermal conductivity kappa (PLA)[W/(m K)]
A_SIDE = 2e-3         # cross-section side a            [m]

# --- Heat exchange ---
K_E = 10.0            # environment heat transfer coeff [W/(m^2 K)]
K_P = 200.0           # heating plate heat transfer coeff [W/(m^2 K)]
T_ENV = 26.85         # environment temperature T_e     [degC]
T_PAD = 60.0          # heating plate temperature T_p   [degC]

# --- Moving boundary: s(t) = S0 + S1*t ---
S0 = 1e-3             # initial length                  [m]
S1 = 1e-3             # boundary speed                  [m/s]
L_END = 0.10          # final length                    [m]
T_BOUND = 240.0       # temperature at x = s(t)         [degC]

# --- Numerics ---
N = 200               # intervals in xi (N+1 nodes)
DT = 0.05             # time step                       [s]
TIME_START = 0.0      # start time                      [s]

# --- Derived constants ---
K_DIFF = KAPPA / (RHO * C_P)                          # k, thermal diffusivity [m^2/s]
VOL_CAP = A_SIDE * RHO * C_P                          # a*rho*c                [J/(m^2 K)]
ALPHA = -(3.0 * K_E + K_P) / VOL_CAP                  # alpha                  [1/s]
BETA = (3.0 * K_E * T_ENV + K_P * T_PAD) / VOL_CAP    # beta                   [K/s]
T_EQ = -BETA / ALPHA                                  # equilibrium temp (no conduction) [degC]

T_HOLD = 120.0                             # simulate this long after the stop [s]
TIME_END = TIME_START + (L_END - S0) / S1 + T_HOLD
N_STEPS = int(round((TIME_END - TIME_START) / DT))
DXI = 1.0 / N

T_STOP = (L_END - S0) / S1 # time when s = L_END    [s]


In [ ]:
import numpy as np

In [ ]:
T_xsi = np.zeros((N_STEPS + 1, N + 1))  # temperature array T(xi, t)
# top row initial condition uniform temperature T_BOUND at t = 0 and small initial size S0
T_xsi[0, :] = T_BOUND

# right side, fixed temperature T_BOUND at x = s(t)
T_xsi[:, -1] = T_BOUND
print(T_xsi)

In [ ]:
def s_of_t(t):
    return np.minimum(S0 + S1 * t, L_END)

def sdot_of_t(t):
    return np.where(S0 + S1 * t < L_END, S1, 0.0)

def get_gamma(xsi, t):
    return xsi * sdot_of_t(t) * DT / (s_of_t(t) * (DXI))

def get_rho(t):
    return K_DIFF * DT / (s_of_t(t) ** 2 * (DXI ** 2))

def build_row_problem(j, T_xsi):
    t = TIME_START + DT * j
    xsi = np.linspace(0, 1, N + 1)
    gamma = get_gamma(xsi, t)
    rho = get_rho(t)
    A = np.zeros((N + 1, N + 1))
    b = np.zeros(N + 1)
    
    main = 1 + 2 * rho + gamma - ALPHA * DT
    lower = -rho * np.ones(N)
    upper = -(rho + gamma[:-1])
    
    main[0] = 1
    main[-1] = 1
    upper[0] = -1
    lower[-1] = 0
    
    b[0] = 0
    b[-1] = T_BOUND

    for i in range(1, N):
        b[i] = T_xsi[j - 1, i] + BETA * DT

    return main, lower, upper, b

def transform_to_x(t, xsi):
    return (S0 + S1 * t) * xsi


import scipy.sparse as sp
from scipy.sparse.linalg import factorized
# for after connecting
def build_periodic_solver(t):
    r = get_rho(t)
    A = sp.diags([-r, 1 + 2 * r - ALPHA * DT, -r], [-1, 0, 1], shape=(N, N), format="lil")
    A[0, N - 1] = -r
    A[N - 1, 0] = -r
    return factorized(A.tocsc())

    

In [ ]:
# solve full system
from scipy.linalg import solve_banded
solve = []
first_after_connect = False
for j in range(1, N_STEPS + 1):
    t = TIME_START + DT * j
    s_dot = sdot_of_t(t)
    if(s_dot != 0.0):
        main, lower, upper, b = build_row_problem(j, T_xsi)
        ab = np.zeros((3, N + 1))
        ab[0, 1:]  = upper
        ab[1, :]   = main
        ab[2, :-1] = lower 
        T_xsi[j] = solve_banded((1, 1), ab, b)
    elif (first_after_connect == False):
        first_after_connect = True
        solve = build_periodic_solver(t)
        T_old = T_xsi[j - 1, :N].copy()
        T_old[0] = 0.5 * (T_xsi[j - 1, 0] + T_xsi[j - 1, N])
        T_new = solve(T_old + BETA * DT)
        T_xsi[j, :N] = T_new
        T_xsi[j, N] = T_new[0]
    else:
        T_new = solve(T_xsi[j - 1, :N] + BETA * DT)
        T_xsi[j, :N] = T_new
        T_xsi[j, N] = T_new[0] 

In [ ]:
# transform the matrix to physical coordinates
import matplotlib.pyplot as plt
def to_physical(T_xsi, x_grid=None):
    t = TIME_START + DT * np.arange(T_xsi.shape[0])
    s = s_of_t(t)
    X = np.outer(s, np.linspace(0, 1, N + 1))
    if x_grid is None:
        return t, X, T_xsi
    T_x = np.full((len(t), len(x_grid)), np.nan)
    for j in range(len(t)):
        inside = x_grid <= s[j]
        T_x[j, inside] = np.interp(x_grid[inside], X[j], T_xsi[j])
    return t, x_grid, T_x

t, X, T = to_physical(T_xsi)

## Plots for the report

In [ ]:
# report style, matches matlab/paperStyle.m (8.85 cm wide, 8 pt Computer Modern, ticks inside)
import matplotlib as mpl
CM = 1 / 2.54
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["cmr10"],
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.top": True,
    "ytick.right": True,
    "lines.linewidth": 1.0,
    "legend.frameon": False,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
})
FIG_W, FIG_H = 8.85 * CM, 8.85 * 0.7 * CM
CMAP = "Reds"                      # sequential: light = cold, dark = hot
IMG_DIR = "../report/images/"
T_MIN, T_MAX = np.nanmin(T_xsi), np.nanmax(T_xsi)   # shared colour scale for both plots

def add_colorbar(fig, ax, mesh):
    cb = fig.colorbar(mesh, ax=ax, pad=0.02)
    cb.set_label(r"temperature $T$ [$^{\circ}\mathrm{C}$]")
    cb.outline.set_linewidth(0.5)
    cb.ax.tick_params(direction="in", width=0.5)
    return cb

In [ ]:
# temperature in physical space, the domain grows with the nozzle position s(t)
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
mesh = ax.pcolormesh(X * 1e3, np.broadcast_to(t[:, None], X.shape), T,
                     shading="gouraud", cmap=CMAP, vmin=T_MIN, vmax=T_MAX, rasterized=True)
ax.plot(s_of_t(t) * 1e3, t, color="k", lw=0.8, label=r"nozzle $s(t)$")
ax.axhline(T_STOP, color="k", lw=0.6, ls="--", label=r"ring closed $t_{\mathrm{stop}}$")
ax.set_xlim(0, L_END * 1e3)
ax.set_ylim(t[0], t[-1])
ax.set_xlabel(r"position $x$ [mm]")
ax.set_ylabel(r"time $t$ [s]")
ax.legend(loc="upper left")
add_colorbar(fig, ax, mesh)
fig.savefig(IMG_DIR + "filament_T_x.pdf")
plt.show()

In [ ]:
# temperature in the front-fixed space, xi = x / s(t) in [0, 1]
xi = np.linspace(0, 1, N + 1)
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
mesh = ax.pcolormesh(xi, t, T_xsi, shading="gouraud", cmap=CMAP,
                     vmin=T_MIN, vmax=T_MAX, rasterized=True)
ax.axhline(T_STOP, color="k", lw=0.6, ls="--", label=r"ring closed $t_{\mathrm{stop}}$")
ax.set_xlim(0, 1)
ax.set_ylim(t[0], t[-1])
ax.set_xlabel(r"transformed position $\xi = x / s(t)$ [-]")
ax.set_ylabel(r"time $t$ [s]")
ax.legend(loc="upper left")
add_colorbar(fig, ax, mesh)
fig.savefig(IMG_DIR + "filament_T_xi.pdf")
plt.show()